# Deep Learning - Brain Tumor Classification
### Approach: Custom Convilutional Neural Network (CNN) from scratch

**Objective:**
Shift from classical machine learning to Deep Learning using PyTorch. We will build a custom CNN, process MRI images in grayscale for efficiency, and use 5-Fold Cross-Validation to rigorously test our architecture.

---
## 1. Environment Setup and Imports
Importing necessary libraries including PyTorch, Torchvision for image processing, and Scikit-Learn for our cross-validation splits. We also verify that our NVIDIA GPU is active and ready.

In [67]:
# Standard library imports for file and array handling
import os
import numpy as np
import matplotlib.pyplot

# Core PyTorch deep learning modules
import torch
from torch import nn
import torch.optim 
import torch.nn.functional as F


# PyTorch data handling and augmentation
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision import transforms


# # Scikit-learn for robust validation strategies and score
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle
from sklearn.metrics import classification_report, confusion_matrix


# Verify hardware acceleration
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


CUDA Available: True
CUDA Version: 12.1
GPU Name: NVIDIA GeForce GTX 1650 Ti


---
## 2. Hardware Configuration & Data Augmentation
To maximize efficiency, we explicitly route all tensor operations to the GPU. 
We also define our image preprocessing pipelines (`torchvision.transforms`). 
- **Training Pipeline**: Includes data augmentation (random flips and rotations) to artificially expand our dataset and prevent the model from memorizing exact pixel layouts (overfitting).
- **Testing Pipeline**: Strictly resizes and normalizes the images. No augmentation is applied to test/validation data to ensure an honest evaluation.
*Note: We convert all RGB MRI scans to single-channel Grayscale to reduce computational overhead without losing critical anatomical data.*

In [68]:
# Configure hardware accelerator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device {device}")


# Define the transformation pipeline for TRAINING images
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),                # Standardize image size for the CNN
    transforms.Grayscale(num_output_channels=1),  # Convert to grayscale (1 channel)
    transforms.RandomHorizontalFlip(),            # Augmentation: flip left/right
    transforms.RandomRotation(10),                # Augmentation: slight rotations
    transforms.ToTensor(),                        # Convert image to mathematical tensor
    transforms.Normalize(mean=[0.5], std=[0.5])   # Scale pixel values to [-1, 1] for stable gradients
])


# Define the transformation pipeline for VALIDATION/TEST images
test_val_transform = transforms.Compose([
    transforms.Resize((224, 224)),                # Standardize image size
    transforms.Grayscale(num_output_channels=1),  # Convert to grayscale
    transforms.ToTensor(),                        # Convert to tensor
    transforms.Normalize(mean=[0.5], std=[0.5])   # Scale using exact same parameters
])

Using device cuda


---
## 3. Data Ingestion & Strict Testing Isolation
We load the raw images directly from the directory structure using `ImageFolder`. 
To prevent **Data Leakage**, we immediately split off 20% of the dataset into a strictly isolated `Test Dataset`. This test set will **never** be used to train or tune the model; it is our final, unbiased exam.
The remaining 80% (`Train/Val Dataset`) will be used for K-Fold Cross-Validation.

In [70]:
# Define path to data directory
DATA_DIR = "D:\projects\Brain Tumor Detection\Brain-Tumor-Detection\data"


# Load ALL images twice (once with augmentation, once clean)
# This allows us to feed clean validation images and augmented training images later
train_dataset_full = ImageFolder(root=DATA_DIR, transform=train_transform)
val_dataset_full = ImageFolder(root=DATA_DIR, transform=test_val_transform)


# Determine dataset sizes
total_size = len(train_dataset_full)
test_size = int(0.2 * total_size)
train_val_size = total_size - test_size

print(f"Total dataset size: {total_size} images")
print(f"Allocated to final Test Set: {test_size} images")
print(f"Allocated to Train/Val (K-Fold): {train_val_size} images")


# Seed the random number generator so the exact same images are isolated for testing every time
torch.manual_seed(42)

# Generate a shuffled list of all indices
all_indices = torch.randperm(total_size).tolist()

# Slice the shuffled list into testing and training/validation index pools
test_indices = all_indices[:test_size]
train_val_indices = all_indices[test_size:]

print(f"Test indices count: {len(test_indices)}")
print(f"TrainVal indices count: {len(train_val_indices)}")


# Lock in the strictly isolated Test Dataset using the clean (val_dataset_full) parameters
test_dataset = Subset(val_dataset_full, test_indices)
train_dataset = Subset(train_dataset, train_val_indices)

print(f"Test dataset size: {len(test_dataset)}")
print(f"Train/Val dataset size: {len(train_dataset_full)}")


# Define the number of folds for the validation strategy
K_FOLDS = 5
kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

print("\nData splitting strategy successfully initialized.")

Total dataset size: 6999 images
Allocated to final Test Set: 1399 images
Allocated to Train/Val (K-Fold): 5600 images
Test indices count: 1399
TrainVal indices count: 5600
Test dataset size: 1399
Train/Val dataset size: 6999

Data splitting strategy successfully initialized.


<>:2: SyntaxWarning: invalid escape sequence '\p'
<>:2: SyntaxWarning: invalid escape sequence '\p'
C:\Users\Malik\AppData\Local\Temp\ipykernel_11680\2114595898.py:2: SyntaxWarning: invalid escape sequence '\p'
  DATA_DIR = "D:\projects\Brain Tumor Detection\Brain-Tumor-Detection\data"


---
## 4. Model Architecture Design
We define a custom Convolutional Neural Network (CNN) class inheriting from `nn.Module`. 
Utilizing blocks of stacked `3x3` convolutions followed by Max Pooling. Stacking two `3x3` convolutions provides the same receptive field as a single `5x5` convolution, but requires fewer parameters and allows for more non-linear activations (ReLUs).

- **Block 1**: Re-levels the 1-channel Grayscale image into 16 complex feature maps, then halves the spatial dimensions.
- **Block 2**: Expands extraction space to 32 feature maps and halves dimensions again to localize broad tumor shapes.
- **Classifier Head**: Flattens the 3D tensor block into a 1D vector to feed into traditional Neural Network layers to calculate final probabilities for 2 classes (Normal / Tumor).


In [71]:
class MyModel(nn.Module):
    """
    A custom Convolutional Neural Network (CNN) architecture designed for 
    binary classification of Brain Tumor MRI scans (Grayscale).
    
    Architecture Breakdown:
    - Block 1: 2x Conv2d (1->16 channels) + ReLU + MaxPool
    - Block 2: 2x Conv2d (16->32 channels) + ReLU + MaxPool
    - Classifier Head: Flatten + Linear(128) + ReLU + Linear(2 output classes)
    """

    def __init__(self):
        super().__init__()

        # Reusable Activation and Pooling modules (since they contain no learnable weights)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # ---------------- BLOCK 1 ----------------
        # Input: [Batch, 1, 224, 224] --> Output: [Batch, 16, 112, 112]
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=16, kernel_size=3, padding=1)
        
        # ---------------- BLOCK 2 ----------------
        # Input: [Batch, 16, 112, 112] --> Output: [Batch, 32, 56, 56]
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)
        
        # ---------------- CLASSIFIER ----------------
        # Compresses 3D tensor to 1D vector and predicts classes
        self.flatten = nn.Flatten()

        # (32 channels * 56 height * 56 width) = 100,352 input features
        self.fc1 = nn.Linear(32 * 56 * 56, 128)
        self.fc2 = nn.Linear(128, 2)


    def forward(self, x):
        """ 
        Defines the forward pass data flow.
        
        Args:
            x (torch.Tensor): Input batch of images, shape (Batch, 1, 224, 224).
            
        Returns:
            torch.Tensor: Raw logits for classification probabilities, shape (Batch, 2).
        """
        
        # Block 1 Flow
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        # Block 2 Flow
        x = self.conv3(x)
        x = self.relu(x)
        x = self.conv4(x)
        x = self.relu(x)
        x = self.pool(x)

        # Flatten & Classify Flow
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        
        return x

# --- VALIDATION TEST ---
# We pass a fake batch of 8 Grayscale images through to ensure mathematical routing is sound
print("Testing architecture tensor routing...")
test_model = MyModel()
fake_batch = torch.randn(8, 1, 224, 224)
output_shape = test_model(fake_batch).shape
print(f"Mathematical Routing Successful. Output shape: {output_shape} (Expected: [8, 2])")        

Testing architecture tensor routing...
Mathematical Routing Successful. Output shape: torch.Size([8, 2]) (Expected: [8, 2])


---
## 5. Cross-Validation Engine
To rigorously validate our model's architecture and hyperparameters without touching our Test Dataset, we employ K-Fold Cross-Validation. 
This function handles:
- **Data Routing**: Ensuring validation datasets are not augmented.
- **State Management**: Initializing a *brand new* model and optimizer for each fold.
- **Early Stopping**: Preventing overfitting by halting training if validation accuracy plateaus.
- **Granular Tracking**: Recording loss and accuracy metrics per epoch to build detailed learning curves.


In [72]:
def train_kfold(model_class, kfold_splitter, train_dataset_full, val_dataset_full, device, epochs=5, batch_size=32, lr=0.001, patience=3):
    """
    Executes K-Fold Cross Validation with Early Stopping.
    
    Args:
        model_class (nn.Module)       :  The Class blueprint of the PyTorch model.
        kfold_splitter (sklearn.KFold):  Initialized KFold index generator.
        train_dataset_full (Dataset)  :  Dataset instance with training augmentations.
        val_dataset_full (Dataset)    :  Dataset instance with clean validation transforms.
        train_val_dataset (Dataset)   :  The 80% array used to generate the split indices.
        device (torch.device)         :  Hardware target ('cuda' or 'cpu').
        epochs (int)                  :  Maximum number of training epochs per fold.
        batch_size (int)              :  Size of batches fed into the DataLoader.
        lr (float)                    :  Learning rate for the Adam optimizer.
        patience (int)                :  Epochs to wait without validation improvement before early stopping.
        
    Returns:
        tuple: (dict of best accuracies per fold, dict of detailed epoch metrics)
    """

    fold_results = {}
    fold_details = {}
    print(f"Starting K-Fold Cross Validation for {epochs} Epochs per fold...")


    # Iterate over the train/val indices provided by Scikit-Learn
    for fold, (train_idx, val_idx) in  enumerate(kfold_splitter.split(train_val_dataset)):

        print(f"\n========== FOLD {fold + 1} ==========")

        # Create isolated subsets for this specific fold
        fold_train_sub = Subset(train_dataset_full, train_idx)
        fold_val_sub = Subset(val_dataset_full, val_idx)

        # Initialize the DataLoaders (batching and shuffling)
        train_loader = DataLoader(fold_train_sub, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(fold_val_sub, batch_size=batch_size, shuffle=False)
        
        # Instantiate a fresh model, loss function, and optimizer
        model = model_class().to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        # Early Stopping and Tracking metrics
        best_fold_accuracy = 0
        best_fold_loss = float('inf')
        best_epoch = 0
        patience_counter = 0

        fold_train_losses, fold_val_losses, fold_accuracies = [], [], []


        for epoch in range(epochs):
            print(f"  [Epoch {epoch+1}/{epochs}]", end=" ")

            # ================= TRAINING PHASE =================
            model.train()
            train_loss = 0.0

            for images, labels in train_loader:
                
                images, labels = images.to(device), labels.to(device)
                
                optimizer.zero_grad()             # Clear old gradients
                outputs = model(images)           # Forward pass
                loss = criterion(outputs, labels) # Calculate error
                loss.backward()                   # Backpropagation
                optimizer.step()                  # Update weights

                train_loss += loss.item() * images.size(0)

            avg_train_loss = train_loss / len(fold_train_sub)
            fold_train_losses.append(avg_train_loss)  


            # ================= VALIDATION PHASE =================
            model.eval()
            val_loss = 0.0
            correct = 0

            with torch.no_grad(): # Disable gradient tracking to save memory
                 for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)  

                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item() * images.size(0)

                    _, prediction = torch.max(outputs, 1) # Extract highest probability class
                    correct += (prediction == labels).sum().item()
            
            avg_val_loss = val_loss / len(fold_val_sub)
            accuracy = 100 * correct / len(fold_val_sub)
            fold_val_losses.append(avg_val_loss)
            fold_accuracies.append(accuracy)


            print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Accuracy: {accuracy:.2f}%")


            # ================= EARLY STOPPING CHECK =================
            if accuracy > best_fold_accuracy:
                best_fold_accuracy = accuracy
                best_fold_loss = avg_val_loss
                best_epoch = epoch + 1
                patience_counter = 0  # Reset patience
                
            else:
                patience_counter += 1
            
            if patience_counter >= patience:
                print(f"  Early stopping triggered at epoch {epoch+1}!")
                break
             

        # Record final stats for this fold     
        fold_results[f"Fold {fold+1}"] = best_fold_accuracy
        fold_details[f"Fold {fold+1}"] = {
            'best_accuracy': best_fold_accuracy,
            'best_loss': best_fold_loss,
            'best_epoch': best_epoch,
            'train_losses': fold_train_losses,
            'val_losses': fold_val_losses,
            'accuracies': fold_accuracies,
            'epochs_completed': len(fold_train_losses)
        }


        print(f"Finished Fold {fold + 1}! Best Validation Accuracy: {best_fold_accuracy:.2f}% (Epoch {best_epoch})")

        
    # ================= OVERALL K-FOLD RESULTS =================
    total_acc = sum(fold_results.values()) / len(fold_results)
    
    print(f"\n==========================================")
    print(f"OVERALL RESULTS OF {len(fold_results)}-FOLD CROSS VALIDATION")
    print(f"==========================================")
    print(f"Average Accuracy: {total_acc:.2f}%")
    print(f"Accuracy per fold:")
    for fold_name, accuracy in fold_results.items():
        print(f"  {fold_name}: {accuracy:.2f}%")
    
    accuracies = list(fold_results.values())
    print(f"\nStatistics:")
    print(f"  Std Deviation: {np.std(accuracies):.2f}%")
    print(f"  Min: {min(accuracies):.2f}%")
    print(f"  Max: {max(accuracies):.2f}%")
    
    return fold_results, fold_details
    


---
## 6. Final Evaluation (Test Dataset)
This function evaluates the fully trained model on the `Test Dataset`—the 20% of data locked away at the very beginning. 
It operates strictly in `model.eval()` mode under a `torch.no_grad()` context to ensure no training occurs.
We utilize `scikit-learn` to dramatically expand our insight, generating a full Classification Report (Precision, Recall, F1-Score) and a Confusion Matrix to see exactly what kind of errors the model made (e.g., False Positives vs. False Negatives).


In [73]:
def prediction(model, test_dataset, device, batch_size=32):
    """
    Evaluates the final trained model on the held-out Test Dataset.
    
    Args:
        model (nn.Module)     : The fully trained PyTorch model.
        test_dataset (Dataset): The subset of images isolated from training.
        device (torch.device) : hardware target ('cuda' or 'cpu').
        batch_size (int)      : Size of batches fed into the DataLoader.
        
    Returns:
        tuple: (final accuracy, list of predictions, list of true labels)
    """


    print("Evaluating model on the held-out Test Dataset...")

    # Initialize isolated DataLoader
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # Put model into strict evaluation mode
    model.eval()


    all_predictions = []
    all_true_labels = []


    # Turn off gradient math to save memory and speed up inference
    with torch.no_grad():
        
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)

            # Identify the predicted class (Normal or Tumor)
            _, predicted = torch.max(outputs, 1)

            # Send results back to CPU for Scikit-Learn evaluation
            all_predictions.extend(predicted.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())


    # Calculate simple accuracy
    correct = sum(p == t for p, t in zip(all_predictions, all_true_labels))
    accuracy = 100 * correct / len(test_dataset)        


    print("\n==========================================")
    print("FINAL TEST DATASET RESULTS")
    print("==========================================")
    print(f"Overall Accuracy: {accuracy:.2f}%\n")
    
    print("Detailed Classification Report:")
    
    # Extract class names directly from the ImageFolder mapping
    class_names = train_dataset_full.classes 
    
    # Print advanced Scikit-Learn metrics
    print(classification_report(all_true_labels, all_predictions, target_names=class_names))
    print("Confusion Matrix:\n", confusion_matrix(all_true_labels, all_predictions))
    
    return accuracy, all_predictions, all_true_labels



---
## 7. Main Execution Flow
This block ties the entire pipeline together. The standard Deep Learning practice used here is:
1. **Hyperparameter Definition**: Set global configurations for training.
2. **K-Fold Validation**: Run the architecture through K-Fold cross-validation to ensure it is stable and verify the hyperparameters are effective.
3. **Champion Model Training**: Discard the validation models and train a unified "champion" model using the *entirety* of the training dataset (all 80%).
4. **Final Exam**: Pass the completely unseen Test Dataset through the champion model for out-of-sample evaluation.

In [74]:
# ==========================================
# 1. SETUP HYPERPARAMETERS
# ==========================================
LEARNING_RATE = 0.001
BATCH_SIZE = 32
K_FOLD_EPOCHS = 10
FINAL_EPOCHS = 8 

# ==========================================
# 2. RUN K-FOLD CROSS VALIDATION
# ==========================================
# Verifies architecture stability and hyperparameter tuning
print("--- STAGE 1: K-FOLD VALIDATION ---")
results, details = train_kfold(
    model_class=MyModel, 
    kfold_splitter=kfold, 
    train_dataset_full=train_dataset_full, 
    val_dataset_full=val_dataset_full, 
    device=device, 
    epochs=K_FOLD_EPOCHS, 
    batch_size=BATCH_SIZE, 
    lr=LEARNING_RATE,
    patience=3
)


# ==========================================
# 3. TRAIN THE FINAL CHAMPION MODEL
# ==========================================
# Using all 5,600 training images simultaneously 
print("\n\n******************************************")
print("TRAINING FINAL CHAMPION MODEL ON ALL DATA")
print("******************************************")


# Initialize fresh model and optimizer
final_model = MyModel().to(device)
criterion = nn.CrossEntropyLoss()
final_optimizer = torch.optim.Adam(final_model.parameters(), lr=LEARNING_RATE)

# Create a DataLoader for the ENTIRE training subset 
# Note: we use train_dataset_full to ensure data augmentation is applied
final_train_loader = DataLoader(train_dataset_full, batch_size=BATCH_SIZE, shuffle=True)


final_model.train()
for epoch in range(FINAL_EPOCHS):
    train_loss = 0.0
    for images, labels in final_train_loader:
        images, labels = images.to(device), labels.to(device)
        
        final_optimizer.zero_grad()
        outputs = final_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        final_optimizer.step()
        train_loss += loss.item() * images.size(0)
        
    print(f"  Final Model - Epoch {epoch+1}/{FINAL_EPOCHS} | Loss: {(train_loss/len(train_dataset_full)):.4f}")


# ==========================================
# 4. FINAL TEST EVALUATION
# ==========================================
print("\n\n--- STAGE 3: OUT-OF-SAMPLE TESTING ---")
test_accuracy, predictions, true_labels = prediction(
    model=final_model, 
    test_dataset=test_dataset, 
    device=device,
    batch_size=BATCH_SIZE
)


--- STAGE 1: K-FOLD VALIDATION ---
Starting K-Fold Cross Validation for 10 Epochs per fold...

========== FOLD 1 ==========
  [Epoch 1/10] Train Loss: 0.1778 | Val Loss: 0.0813 | Val Accuracy: 96.88%
  [Epoch 2/10] Train Loss: 0.0753 | Val Loss: 0.0563 | Val Accuracy: 98.48%
  [Epoch 3/10] Train Loss: 0.0573 | Val Loss: 0.0600 | Val Accuracy: 98.21%
  [Epoch 4/10] Train Loss: 0.0431 | Val Loss: 0.0636 | Val Accuracy: 98.84%
  [Epoch 5/10] Train Loss: 0.0372 | Val Loss: 0.0590 | Val Accuracy: 97.86%
  [Epoch 6/10] Train Loss: 0.0253 | Val Loss: 0.0480 | Val Accuracy: 98.84%
  [Epoch 7/10] Train Loss: 0.0290 | Val Loss: 0.0398 | Val Accuracy: 98.84%
  Early stopping triggered at epoch 7!
Finished Fold 1! Best Validation Accuracy: 98.84% (Epoch 4)

========== FOLD 2 ==========
  [Epoch 1/10] Train Loss: 0.2311 | Val Loss: 0.1561 | Val Accuracy: 94.02%
  [Epoch 2/10] Train Loss: 0.0975 | Val Loss: 0.1064 | Val Accuracy: 96.34%
  [Epoch 3/10] Train Loss: 0.0677 | Val Loss: 0.1005 | Val Accu